# Aula 02 — Estatística Descritiva

## Objetivos
1. Calcular e interpretar **medidas de tendência central** (média, mediana, moda).
2. Calcular e interpretar **medidas de dispersão** (variância, desvio-padrão, IQR).
3. Entender **quantis** e seu uso.
4. Aplicar tudo aos dados do IPCA e do PIB per capita por UF.

---

## 1. Tendência Central

Tendência central responde: *"qual valor representa o conjunto?"*

### 1.1 Média Aritmética

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$$

- Soma dos valores dividida pelo número de observações.
- Sensível a **outliers** (valores extremos).

### 1.2 Mediana

O valor que ocupa a **posição central** quando os dados estão ordenados.
- Para $n$ ímpar: elemento na posição $(n+1)/2$.
- Para $n$ par: média dos dois centrais.
- **Robusta** a outliers.

### 1.3 Moda

Valor (ou valores) que aparece(m) com **maior frequência**.
- Pode haver mais de uma (distribuição bimodal/multimodal).
- Útil em dados categóricos.

### Quando usar cada uma?

| Situação | Use |
|---|---|
| Distribuição simétrica, sem outliers | Média |
| Distribuição assimétrica, com outliers | Mediana |
| Dados categóricos | Moda |

---

## 2. Medidas de Dispersão

Tendência central sozinha engana. Considere:
- Conjunto A: `[5, 5, 5, 5, 5]` → média = 5
- Conjunto B: `[0, 0, 5, 10, 10]` → média = 5

Mesma média, **realidades completamente diferentes**. Por isso medimos a *dispersão*.

### 2.1 Amplitude
$$\text{amplitude} = \max(x) - \min(x)$$

### 2.2 Variância
$$s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

> **Por que dividir por $n-1$?** É a *correção de Bessel*. Usamos $n-1$ em vez de
> $n$ porque a variância amostral subestima a populacional. Esse ajuste a torna
> um estimador **não-viesado**.

### 2.3 Desvio-padrão
$$s = \sqrt{s^2}$$

Na mesma unidade dos dados — mais fácil de interpretar que a variância.

### 2.4 Intervalo Interquartílico (IQR)
$$\text{IQR} = Q_3 - Q_1$$

- $Q_1$ = quartil 1 (25%); $Q_3$ = quartil 3 (75%).
- Robusto a outliers.
- Base do **boxplot** (aula 03).

In [ ]:
# === SETUP PARA GOOGLE COLAB ===
# Esta célula garante que o notebook funcione tanto em ambiente local
# quanto em quem clonar o repositório direto no Colab.

import sys, os, subprocess

# 1) Instala dependências (silencioso se já existirem)
pkgs = ["numpy", "pandas", "matplotlib", "seaborn", "scipy", "requests", "plotly", "statsmodels"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# 2) Se estivermos no Colab e o repo ainda não foi clonado, clona.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/220719/curso-estatistica-python.git"  # ajuste após criar o repo

if IN_COLAB and not os.path.exists("/content/curso-estatistica-python"):
    subprocess.run(["git", "clone", REPO_URL, "/content/curso-estatistica-python"], check=False)

# 3) Adiciona o diretório utils ao PYTHONPATH para importar o módulo SIDRA
BASE = "/content/curso-estatistica-python" if IN_COLAB else ".."
if BASE not in sys.path:
    sys.path.append(BASE)

# 4) Pasta de gráficos (cria se não existir)
GRAFICOS_DIR = os.path.join(BASE, "graficos")
os.makedirs(GRAFICOS_DIR, exist_ok=True)

print("✓ Ambiente pronto. Pasta de gráficos:", GRAFICOS_DIR)

In [ ]:
import numpy as np
import pandas as pd
from utils.sidra_api import obter_ipca_mensal, obter_pib_per_capita_uf

# Carrega IPCA dos últimos 60 meses
df_ipca = obter_ipca_mensal(60)
ipca = df_ipca["variacao_mensal"]

print(f"Total de observações: {len(ipca)}")
ipca.head()

### Cálculo manual vs. Pandas

Vamos calcular tudo "na mão" primeiro para entender, depois usar o pandas.

In [ ]:
# Cálculo MANUAL (didático)
n = len(ipca)
media_manual = sum(ipca) / n
variancia_manual = sum((x - media_manual)**2 for x in ipca) / (n - 1)
desvio_manual = variancia_manual ** 0.5

print(f"Média manual:        {media_manual:.4f}")
print(f"Variância manual:    {variancia_manual:.4f}")
print(f"Desvio-padrão manual: {desvio_manual:.4f}")

In [ ]:
# Cálculo com PANDAS (na prática usamos sempre)
print(f"Média:          {ipca.mean():.4f}")
print(f"Mediana:        {ipca.median():.4f}")
print(f"Moda:           {ipca.mode().values}")
print(f"Variância:      {ipca.var():.4f}")
print(f"Desvio-padrão:  {ipca.std():.4f}")
print(f"Mínimo:         {ipca.min():.4f}")
print(f"Máximo:         {ipca.max():.4f}")
print(f"Amplitude:      {ipca.max() - ipca.min():.4f}")
print(f"Q1 (25%):       {ipca.quantile(0.25):.4f}")
print(f"Q3 (75%):       {ipca.quantile(0.75):.4f}")
print(f"IQR:            {ipca.quantile(0.75) - ipca.quantile(0.25):.4f}")

### Interpretando os resultados

- **Média ≠ Mediana?** Sinal de assimetria. Média > Mediana → cauda à direita.
- **Desvio-padrão grande relativo à média** → dados voláteis.
- O método `.describe()` traz tudo isso em uma chamada.

In [ ]:
ipca.describe()

## 3. Comparando distribuições: PIB per capita por UF

Agora um exemplo com **dados muito assimétricos**: o PIB per capita das 27 UFs.

In [ ]:
df_pib = obter_pib_per_capita_uf()
df_pib = df_pib.sort_values("pib_per_capita", ascending=False).reset_index(drop=True)
df_pib

In [ ]:
pib = df_pib["pib_per_capita"]

print(f"Média:    R$ {pib.mean():,.2f}")
print(f"Mediana:  R$ {pib.median():,.2f}")
print(f"Desvio:   R$ {pib.std():,.2f}")
print(f"\nObserve: média > mediana → distribuição assimétrica à direita")
print(f"(UFs ricas — DF, SP — puxam a média para cima)")

## 4. Coeficiente de Variação (CV)

$$CV = \frac{s}{\bar{x}} \times 100\%$$

Permite **comparar dispersões** entre conjuntos com unidades ou magnitudes diferentes.

In [ ]:
cv_ipca = ipca.std() / abs(ipca.mean()) * 100
cv_pib  = pib.std() / pib.mean() * 100

print(f"CV do IPCA mensal:        {cv_ipca:.1f}%")
print(f"CV do PIB per capita UF:  {cv_pib:.1f}%")
print(f"\nQuanto maior o CV, maior a variabilidade relativa.")

## 5. Visualizando: gráfico de barras com a média

In [ ]:
import matplotlib.pyplot as plt
import os

fig, ax = plt.subplots(figsize=(12, 6))
cores = ["#2ca02c" if v >= pib.median() else "#d62728" for v in df_pib["pib_per_capita"]]
ax.barh(df_pib["uf"], df_pib["pib_per_capita"], color=cores)
ax.axvline(pib.mean(),   color="blue",  linestyle="--", label=f"Média: R$ {pib.mean():,.0f}")
ax.axvline(pib.median(), color="black", linestyle=":",  label=f"Mediana: R$ {pib.median():,.0f}")
ax.set_xlabel("PIB per capita (R$)")
ax.set_title("PIB per capita por UF — média vs mediana", fontweight="bold")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, "aula02_pib_uf_barras.png"), dpi=150, bbox_inches="tight")
plt.show()

## Exercícios

1. Calcule média, mediana e desvio-padrão do IPCA **somente para os últimos 12 meses**.
2. Qual UF tem PIB per capita mais próximo da mediana nacional?
3. Crie uma função `resumo(serie)` que devolva um dicionário com média, mediana, desvio e IQR.

---

## Resumo

| Conceito | Quando usar |
|---|---|
| Média | Distribuições simétricas |
| Mediana | Dados assimétricos / com outliers |
| Desvio-padrão | Dispersão na mesma unidade dos dados |
| IQR | Dispersão robusta |
| CV | Comparar dispersões de escalas diferentes |

**Próxima aula:** Visualização exploratória — histogramas, boxplots, densidade.